# Урок 10. Overfitting и Underfitting
**Библиотеки:** numpy, sklearn, matplotlib

**Цель:** научиться видеть переобучение по разрыву train/test и по графику.

## Простыми словами
- **Underfitting** — модель-двоечник: слишком простая, плохо и на train, и на test.
- **Overfitting** — модель-зубрила: запомнила train наизусть (вместе с шумом!),
  на train блестяще, на новых данных — провал.
- **Bias** — склонность к слишком простым ответам (недообучение).
  **Variance** — чрезмерная чувствительность к конкретным данным (переобучение).
  
Диагноз ставим по двум числам: train error и test error.
- Оба плохие → underfitting. | Train отличный, test плохой → overfitting. | Оба хорошие → норм.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

rng = np.random.default_rng(5)
X = np.sort(rng.uniform(0, 10, 80)).reshape(-1, 1)
y = np.sin(X[:, 0]) * 3 + X[:, 0] + rng.normal(0, 0.7, 80)   # волна + шум

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

xs = np.linspace(0, 10, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, depth in zip(axes, [1, 4, 20]):
    m = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_tr, y_tr)
    tr, te = r2_score(y_tr, m.predict(X_tr)), r2_score(y_te, m.predict(X_te))
    ax.scatter(X_tr, y_tr, s=12, alpha=0.6)
    ax.plot(xs, m.predict(xs), 'r')
    ax.set_title(f'depth={depth}\ntrain R2={tr:.2f} | test R2={te:.2f}')
plt.show()

In [ ]:
# График сложность модели -> качество (главный график урока!)
depths = range(1, 21)
tr_scores, te_scores = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=0).fit(X_tr, y_tr)
    tr_scores.append(r2_score(y_tr, m.predict(X_tr)))
    te_scores.append(r2_score(y_te, m.predict(X_te)))

plt.plot(depths, tr_scores, 'o-', label='train R2')
plt.plot(depths, te_scores, 'o-', label='test R2')
plt.xlabel('сложность модели (max_depth)'); plt.ylabel('R2')
plt.legend(); plt.grid(True)
plt.title('Train растёт всегда. Test растёт, потом ПАДАЕТ = переобучение'); plt.show()

## Что произошло
depth=1: прямая ступенька — недообучение (плохо везде). depth=4: гладко повторяет волну — отлично.
depth=20: линия дёргается за каждой шумной точкой — зубрёжка; train R2 ≈ 1.0, test упал.
На втором графике видно «золотую середину»: test-кривая растёт, достигает пика и падает.

Лечение overfitting: проще модель / больше данных / регуляризация (урок 11).
Лечение underfitting: сложнее модель / больше признаков / polynomial features.

## Типичные ошибки
1. Радоваться train R2 = 0.99 — это часто симптом болезни, а не успех.
2. Выбирать сложность по train — выбирай по test (а лучше по validation/cross-validation).
3. Думать, что больше параметров = всегда лучше.

## Конспект 📓
Underfit: плохо везде. Overfit: train отлично, test плохо. Диагноз = разрыв train/test.
График сложность→качество: train растёт всегда, test растёт и падает. Бери пик test.

---
## Мини-задание 💪
Увеличь шум в данных (0.7 → 2.0) и перестрой второй график. Сместился ли оптимальный depth? В какую сторону и почему?

## 5 проверочных вопросов ❓
1. Опиши overfitting и underfitting через аналогию со студентом.
2. Train R2=0.99, test R2=0.55 — диагноз? Лечение?
3. Train R2=0.4, test R2=0.38 — диагноз? Лечение?
4. Почему train-качество растёт с любым усложнением модели?
5. Что такое bias и variance простыми словами?

## Домашнее задание 🏠
Задачи 24–25 из practice_tasks.md. Повтори эксперимент с PolynomialFeatures + LinearRegression вместо дерева (degree от 1 до 15).
